# 03 — Persistence Filters and Retrospective Chronological Stability

**Project:** Volatility & Dispersion (V&D) Cloud — personal research project
**Phase:** 2B — tickets G10–G11
**Data:** public daily prices via `yfinance` only (same developed-market country ETF surface as
`02_cross_market_validation_clean.ipynb`)

**Why this notebook.** `02_cross_market_validation_clean.ipynb` ended with a mixed, unresolved
verdict: descriptive regime separation existed, but it didn't clearly survive a baseline challenge
or a stacked permutation-plus-HAC robustness check. Two questions were left open:

1. Does requiring a regime to *persist* for several consecutive days before counting it — rather
   than using same-date, potentially flickery raw labels — produce a cleaner, more stable picture?
2. Do descriptive relationships from 2015–2020 retain direction or magnitude in the later 2021–2026 period? The public record does not establish a freeze of model choices before that period.

This notebook computes both comparisons. Verified revised conclusions appear below.

**Data.** Public daily adjusted prices via `yfinance`.

**Verified execution.** The revised code ran on real Yahoo data on October 13, 2026 (America/Los_Angeles), under Python 3.12.1 on Windows. All 22 tests and cached-data invariant checks passed; online/offline output hashes matched. This narrative was updated afterward without changing code cells. Result tables are exported in `results/`; interactive notebook outputs remain cleared.

## 1. Rebuild the G1–G4 Foundation

This notebook assumes the same EFA / URTH / 13-country-ETF surface, signal variants, and same-date
cloud geometry built in `02_cross_market_validation_clean.ipynb` (tickets G1–G4). The setup is
repeated here in compressed form so this notebook can run standalone.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf



REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
import config
from src.vdcloud import (adjusted_close, price_returns, rolling_zscore, rolling_midpoint,
                         build_cloud, forward_return, purge_cross_boundary,
                         persistence_confirmed, prior_quantile_buckets, bucket_inference,
                         label_null_test, classify_zone)
from src.repro import download


PRIMARY_BENCHMARK = config.DM_PRIMARY_BENCHMARK   # EFA
COMPONENT_TICKERS = config.DM_COUNTRY_ETF_UNIVERSE
ALL_TICKERS = [PRIMARY_BENCHMARK, config.DM_CONTEXT_BENCHMARK] + COMPONENT_TICKERS

Z_WINDOW, VOL_WINDOW, SMOOTH_WINDOW = config.Z_WINDOW, config.VOL_WINDOW, config.SMOOTH_WINDOW
CLOUD_FAST, CLOUD_MEDIUM, CLOUD_SLOW = config.CONVERSION_WINDOW, config.BASE_WINDOW, config.SPAN_B_WINDOW
FORWARD_HORIZONS = config.FORWARD_HORIZONS
MIN_BUCKET_N = config.MIN_BUCKET_N
PERSISTENCE_WINDOWS = config.PERSISTENCE_WINDOWS   # [3, 5]

raw = download(tickers=ALL_TICKERS, start=config.START_DATE, end=config.END_DATE,
                   auto_adjust=True, progress=True)
prices = adjusted_close(raw, ALL_TICKERS)
returns = price_returns(prices)

efa_returns = returns[PRIMARY_BENCHMARK].copy()
dm_country_returns = returns[COMPONENT_TICKERS].copy()
available_count = dm_country_returns.notna().sum(axis=1)

ANNUALIZATION_FACTOR = np.sqrt(252)
dm_country_dispersion_ann = (dm_country_returns.std(axis=1, skipna=True, ddof=1)
                              .where(available_count >= 10)) * ANNUALIZATION_FACTOR
efa_realized_vol = efa_returns.rolling(window=VOL_WINDOW, min_periods=VOL_WINDOW).std(ddof=1) * ANNUALIZATION_FACTOR


# rolling_zscore imported from src.vdcloud


dm_country_dispersion_z_smooth = rolling_zscore(dm_country_dispersion_ann, Z_WINDOW).rolling(window=SMOOTH_WINDOW, min_periods=SMOOTH_WINDOW).mean()
efa_realized_vol_z_smooth = rolling_zscore(efa_realized_vol, Z_WINDOW).rolling(window=SMOOTH_WINDOW, min_periods=SMOOTH_WINDOW).mean()

signal_variants = {
    "additive": 0.70 * dm_country_dispersion_z_smooth + 0.30 * efa_realized_vol_z_smooth,
    "penalty": 0.70 * dm_country_dispersion_z_smooth - 0.30 * efa_realized_vol_z_smooth,
    "dispersion_only": dm_country_dispersion_z_smooth.copy(),
    "volatility_only": efa_realized_vol_z_smooth.copy(),
}


# rolling_midpoint imported from src.vdcloud


def build_same_date_cloud_regime(signal):
    return build_cloud(signal, CLOUD_FAST, CLOUD_MEDIUM, CLOUD_SLOW)["regime_state"]


raw_regimes = {name: build_same_date_cloud_regime(s) for name, s in signal_variants.items()}


def compute_forward_return(return_series, horizon):
    return forward_return(return_series, horizon)


efa_fwd = pd.DataFrame({f"fwd_{h}d": compute_forward_return(efa_returns, h) for h in FORWARD_HORIZONS})

efa_fwd = purge_cross_boundary(efa_fwd, config.IN_SAMPLE_END)

print("Foundation rebuilt; split-crossing outcomes purged.")
for name, s in raw_regimes.items():
    print(f"  {name:18s} valid regime obs: {s.notna().sum()}")

## 2. Persistence-Confirmation Logic (G10)

Same-date regime labels can flicker day to day. `create_persistence_confirmed_regime` labels date
*t* with the raw regime only if that same regime has held for the prior `window - 1` trading days
**and** date *t* itself — i.e. `window` consecutive identical, valid raw-regime observations ending
at *t*. Any gap, change, or missing value inside that window yields `"unconfirmed"`. Confirmed
labels are **not** forward-filled into unconfirmed dates — this is a strict, same-or-no-label
confirmation rule.

Two windows are tested: 3-day and 5-day confirmation.

In [ ]:
UNCONFIRMED_LABEL = "unconfirmed"
REGIME_STATES = ["above_cloud", "inside_cloud", "below_cloud"]


def create_persistence_confirmed_regime(regime_series, window):
    return persistence_confirmed(regime_series, window, UNCONFIRMED_LABEL)


confirmed_regimes = {}
for name, raw_series in raw_regimes.items():
    for window in PERSISTENCE_WINDOWS:
        confirmed_regimes[(name, window)] = create_persistence_confirmed_regime(raw_series, window)

# Worked example matching the confirmation rule
example_a = pd.Series(["above_cloud", "above_cloud", "above_cloud"])
example_b = pd.Series(["above_cloud", "above_cloud", "inside_cloud"])
print("3D example (above, above, above)  ->", create_persistence_confirmed_regime(example_a, 3).tolist())
print("3D example (above, above, inside) ->", create_persistence_confirmed_regime(example_b, 3).tolist())

In [ ]:
churn_rows = []
for name, raw_series in raw_regimes.items():
    raw_series = raw_series.dropna()
    raw_transitions = raw_series.ne(raw_series.shift()).iloc[1:].sum()
    raw_rate = 100.0 * raw_transitions / len(raw_series)
    churn_rows.append({"variant": name, "regime_source": "raw", "transition_rate_per_100_obs": raw_rate,
                        "percent_unconfirmed": 0.0})
    for window in PERSISTENCE_WINDOWS:
        confirmed = confirmed_regimes[(name, window)].reindex(raw_series.index)
        transitions = confirmed.ne(confirmed.shift()).iloc[1:].sum()
        rate = 100.0 * transitions / len(confirmed)
        pct_unconfirmed = 100.0 * (confirmed == UNCONFIRMED_LABEL).mean()
        churn_rows.append({"variant": name, "regime_source": f"confirmed_{window}d",
                            "transition_rate_per_100_obs": rate, "percent_unconfirmed": pct_unconfirmed})

churn_diagnostics = pd.DataFrame(churn_rows)
churn_diagnostics

Confirmation reduces the number of labeled observations and introduces unconfirmed states. Its effect on transition counts is an empirical diagnostic, not guaranteed: entries into and exits from unconfirmed states are counted as transitions.


In [ ]:
def summarize_regime_forward_returns(regime_series, return_frame, source_label, variant_label):
    joined = pd.DataFrame({"regime_state": regime_series}).join(return_frame, how="left")
    rows = []
    for h in FORWARD_HORIZONS:
        col = f"fwd_{h}d"
        uncond_mean = joined.loc[joined["regime_state"].notna(), col].dropna().mean()
        for regime, group in joined.dropna(subset=["regime_state", col]).groupby("regime_state"):
            r = group[col]
            rows.append({"variant": variant_label, "regime_source": source_label, "regime": regime,
                         "horizon": h, "count": len(r), "mean_return": r.mean(),
                         "excess_vs_unconditional": r.mean() - uncond_mean})
    return rows


persistence_forward_rows = []
for name, raw_series in raw_regimes.items():
    persistence_forward_rows += summarize_regime_forward_returns(raw_series, efa_fwd, "raw", name)
    for window in PERSISTENCE_WINDOWS:
        confirmed = confirmed_regimes[(name, window)]
        persistence_forward_rows += summarize_regime_forward_returns(confirmed, efa_fwd, f"confirmed_{window}d", name)

persistence_forward_return_summary = pd.DataFrame(persistence_forward_rows)
persistence_forward_return_summary.head(12)

Both raw and five-day-confirmed labels are carried into the chronological comparison. Choosing a preferred label set after inspecting later-period results is retrospective analysis, not validation on an untouched period. The verified persistence comparison is reported below.


## 3. Retrospective Chronological Split

Origin dates through 2020-12-31 form the earlier period; dates from 2021-01-01 through 2026-05-29 form the later period. For each horizon, earlier-period outcomes are excluded if their endpoints cross the split. Features may use prior history when entering the later period.

Excess returns are computed relative to each period's unconditional mean over observations with an available label and outcome. The later period is not claimed to have been untouched during model design.


In [ ]:
IN_SAMPLE_END = pd.Timestamp(config.IN_SAMPLE_END)
HOLDOUT_START = pd.Timestamp(config.HOLDOUT_START)

period = pd.Series(np.where(efa_fwd.index <= IN_SAMPLE_END, "in_sample",
                    np.where(efa_fwd.index >= HOLDOUT_START, "holdout", "excluded")),
                    index=efa_fwd.index)

print(period.value_counts())
print(f"\nIn-sample : {efa_fwd.index[period == 'in_sample'].min().date()} to {efa_fwd.index[period == 'in_sample'].max().date()}")
print(f"Holdout   : {efa_fwd.index[period == 'holdout'].min().date()} to {efa_fwd.index[period == 'holdout'].max().date()}")

In [ ]:
def period_regime_forward_summary(regime_series, source_label, variant_label):
    joined = pd.DataFrame({"regime_state": regime_series, "period": period}).join(efa_fwd, how="left")
    rows = []
    for p in ["in_sample", "holdout"]:
        sub = joined[joined["period"] == p]
        for h in FORWARD_HORIZONS:
            col = f"fwd_{h}d"
            uncond_mean = sub.loc[sub["regime_state"].notna(), col].dropna().mean()
            for regime, group in sub.dropna(subset=["regime_state", col]).groupby("regime_state"):
                r = group[col]
                rows.append({"period": p, "regime_source": source_label, "variant": variant_label,
                             "regime": regime, "horizon": h, "count": len(r),
                             "excess_vs_unconditional": r.mean() - uncond_mean})
    return rows


period_forward_rows = []
for name, raw_series in raw_regimes.items():
    period_forward_rows += period_regime_forward_summary(raw_series, "raw", name)
    period_forward_rows += period_regime_forward_summary(confirmed_regimes[(name, 5)], "confirmed_5d", name)

regime_period_forward_return_summary = pd.DataFrame(period_forward_rows)
print(f"Total in-sample/holdout cells: {len(regime_period_forward_return_summary)}")

In [ ]:
NEAR_ZERO_EXCESS = 1e-4

def lookup_period_cell(source, variant, regime, horizon, p):
    row = regime_period_forward_return_summary[
        (regime_period_forward_return_summary["period"] == p)
        & (regime_period_forward_return_summary["regime_source"] == source)
        & (regime_period_forward_return_summary["variant"] == variant)
        & (regime_period_forward_return_summary["regime"] == regime)
        & (regime_period_forward_return_summary["horizon"] == horizon)
    ]
    if row.empty:
        return None, None
    r = row.iloc[0]
    return int(r["count"]), float(r["excess_vs_unconditional"])


consistency_rows = []
for source in ["raw", "confirmed_5d"]:
    for variant in signal_variants:
        candidate_regimes = REGIME_STATES + ([UNCONFIRMED_LABEL] if source != "raw" else [])
        for regime in candidate_regimes:
            for horizon in [21, 63]:
                is_n, is_excess = lookup_period_cell(source, variant, regime, horizon, "in_sample")
                ho_n, ho_excess = lookup_period_cell(source, variant, regime, horizon, "holdout")
                is_n, ho_n = is_n or 0, ho_n or 0
                low_n = is_n < MIN_BUCKET_N or ho_n < MIN_BUCKET_N

                if low_n or is_excess is None or ho_excess is None:
                    flag = "low_n"
                elif abs(is_excess) < NEAR_ZERO_EXCESS:
                    flag = "inconclusive"
                else:
                    same_sign = np.sign(is_excess) == np.sign(ho_excess)
                    decay = abs(ho_excess) / abs(is_excess)
                    if same_sign and decay >= 0.5:
                        flag = "consistent"
                    elif same_sign:
                        flag = "directionally_consistent_but_weaker"
                    else:
                        flag = "sign_flipped"

                consistency_rows.append({"regime_source": source, "variant": variant, "regime": regime,
                                          "horizon": horizon, "in_sample_count": is_n, "holdout_count": ho_n,
                                          "in_sample_excess": is_excess, "holdout_excess": ho_excess,
                                          "consistency_flag": flag})

in_sample_vs_holdout_consistency = pd.DataFrame(consistency_rows)
print(in_sample_vs_holdout_consistency["consistency_flag"].value_counts())

**Verified revised counts:** the 56-cell grid contains 35 sign flips, 11 consistent, 6 weaker same-sign cells, and 4 low-count cells. Of 48 named-regime cells, 28 flipped, 11 were consistent, 5 were weaker same-sign, and 4 were low-count. Unconfirmed cells are reported separately: seven flips and one weaker same-sign result.


## 4. Does Persistence Confirmation Improve Temporal Stability?

For each variant/regime/horizon cell, compare the `raw` and `confirmed_5d` consistency flags
side by side and identify which label set is preferred on temporal-stability grounds — never on
which one happened to show the larger in-sample excess, since that would just relocate the
overfitting problem from regime definition to label-set selection.

In [ ]:
FLAG_RANK = {"consistent": 0, "directionally_consistent_but_weaker": 1,
             "inconclusive": 2, "sign_flipped": 3, "low_n": 4}

def consistency_row(source, variant, regime, horizon):
    row = in_sample_vs_holdout_consistency[
        (in_sample_vs_holdout_consistency["regime_source"] == source)
        & (in_sample_vs_holdout_consistency["variant"] == variant)
        & (in_sample_vs_holdout_consistency["regime"] == regime)
        & (in_sample_vs_holdout_consistency["horizon"] == horizon)
    ]
    return row.iloc[0] if len(row) else None


preference_counts = {"raw": 0, "confirmed_5d": 0, "neither": 0, "inconclusive": 0}
for variant in signal_variants:
    for regime in REGIME_STATES:
        for horizon in [21, 63]:
            raw_row = consistency_row("raw", variant, regime, horizon)
            c5_row = consistency_row("confirmed_5d", variant, regime, horizon)
            if raw_row is None or c5_row is None:
                continue
            raw_rank, c5_rank = FLAG_RANK[raw_row["consistency_flag"]], FLAG_RANK[c5_row["consistency_flag"]]
            if raw_rank < c5_rank:
                preference_counts["raw"] += 1
            elif c5_rank < raw_rank:
                preference_counts["confirmed_5d"] += 1
            elif raw_row["consistency_flag"] in ("sign_flipped", "inconclusive"):
                preference_counts["neither"] += 1
            else:
                preference_counts["inconclusive"] += 1

print("Preferred label set across variant/regime/horizon comparisons:")
for label, count in preference_counts.items():
    print(f"  {label:14s}: {count}")

**Verified revised persistence comparison:** among 24 matched comparisons, raw labels are preferred in 6, confirmed labels in 4, neither in 11, and the rule is inconclusive in 3. These are retrospective descriptive preferences, not independent tests.


## Chronological Stability Verdict — Verified Revised Run

Among 48 named-regime cells (four lenses × three regimes × two label sources × two horizons), 28 sign-flipped, 11 met the consistency rule, 5 retained a weaker same-sign excess, and 4 had insufficient observations. Across 44 adequately sized named-regime cells, 16 retained their sign. Eight unconfirmed cells are separate: seven flipped and one retained a weaker same-sign excess.

The full 56-cell grid therefore has 35 sign flips, 11 consistent, 6 directionally consistent but weaker, and 4 low-count cells. Earlier-period forward outcomes crossing the split are purged by horizon. The consistent category requires at least 50% retained absolute excess; the weaker category does not.

Raw labels are retrospectively preferred in 6 of 24 matched comparisons, confirmed labels in 4, neither in 11, and the rule is inconclusive in 3. Persistence offers no broad improvement. Cells are dependent, choices were not demonstrably frozen before the later period, and same-sign excess alone is not evidence against noise or of predictive validity.
